In [ ]:
import digitalhub as dh
import pandas as pd
import requests
import os
!pip install seaborn
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT = "test-otp-v2"
project = dh.get_or_create_project(PROJECT)

zoi: str = "allBologna"  ## Choose between "allBologna" and "restrictedAv"
method: str = "simplified" ## Choose between "simplified" and "extended"
version: str = "20250926"

In [ ]:
df_car_trans_di = project.get_dataitem(f"otp_results_{method}_{zoi}_CAR_PARK_TRANSIT_v{version}")
df_car_trans = df_car_trans_di.as_df()

df_car_di = project.get_dataitem(f"otp_results_{method}_{zoi}_CAR_v{version}")
df_car = df_car_di.as_df()

df_car_park_di = project.get_dataitem(f"otp_results_{method}_{zoi}_CAR_PARK_v{version}")
df_car_park = df_car_park_di.as_df()

df_trans_di = project.get_dataitem(f"otp_results_{method}_{zoi}_TRANSIT_v{version}")
df_trans = df_trans_di.as_df()

print(df_car.columns)

# CAR E CAR_PARK

In [ ]:
# Confronto i df in car con e senza park
df_compare = pd.merge(
    df_car_park, 
    df_car, 
    on=['row_index', 'origin_lat', 'origin_lon', 'from', 'dest_lat', 'dest_lon','to'], 
    how='outer',
    suffixes=["_carPark","_car"])

print("N. NAs in car:", int(df_compare["duration_seconds_car"].isna().sum()))
print("N. NAs in carPark:", int(df_compare["duration_seconds_carPark"].isna().sum()))

print("N. success in car:", sum(df_compare["status_car"]=="success"))
print("N. success in carPark:", sum(df_compare["status_carPark"]=="success"))

df_compare.head()

In [ ]:
sns.pairplot(
    data=df_compare[["duration_seconds_car", "duration_seconds_carPark",
                        "walk_time_seconds_car", "walk_time_seconds_carPark",
                        "number_of_legs_car", "number_of_legs_carPark"]],
    diag_kind="hist"
)
    

In [ ]:
print("Min diff:", min(df_compare["duration_seconds_car"] - df_compare["duration_seconds_carPark"]))
print("Max diff:", max(df_compare["duration_seconds_car"] - df_compare["duration_seconds_carPark"]))
print("Mean diff:", sum(df_compare["duration_seconds_car"] - df_compare["duration_seconds_carPark"])/df_compare.shape[0])

In [ ]:
df_compare["CAR_car"] = 0.0
df_compare["WALK_car"] = 0.0
for i, (durations, modes) in enumerate(zip(df_compare["transport_durations_car"], df_compare["transport_modes_car"])):
    for d, m in zip(durations, modes):
        m_mod = f"{m}_car"
        df_compare.loc[i, m_mod] = d
df_compare["CAR_carPark"] = 0.0
df_compare["WALK_carPark"] = 0.0
for i, (durations, modes) in enumerate(zip(df_compare["transport_durations_carPark"], df_compare["transport_modes_carPark"])):
    for d, m in zip(durations, modes):
        m_mod = f"{m}_carPark"
        df_compare.loc[i, m_mod] = d

# Scatterplot
p = sns.scatterplot(data=df_compare, y="CAR_car", x="WALK_car")
p = sns.scatterplot(data=df_compare, y="CAR_carPark", x="WALK_carPark")
p = plt.xlabel("WALK time")
p = plt.ylabel("CAR time")


In [ ]:
df_car["waiting_time"] = df_car["duration_seconds"] - df_car["transport_durations"].apply(lambda x: sum(x))
print(df_car["waiting_time"].describe())

df_car_park["waiting_time"] = df_car_park["duration_seconds"] - df_car_park["transport_durations"].apply(lambda x: sum(x))
print(df_car_park["waiting_time"].describe())

df_car_park

# TRANSIT_CAR_PARK

In [ ]:
# Check, when car_park and transit, the % of times that transit is choosen -> the bus is never chosen!
all_modes = set([])
for modes in df_car_trans["transport_modes"]:
    for m in modes:
        if m not in all_modes:
            all_modes.add(m)
all_modes

# TRANSIT

In [ ]:
# Check transit only
print("N. elements:", df_trans.shape[0])
print("N. elements non successful:", df_trans[df_trans["status"]!="success"].shape[0])
df_trans_nona = df_trans[(df_trans["status"]=="success") & ~(df_trans["row_index"].isna())]
print("N. new elements:", df_trans_nona.shape[0])

In [ ]:
df_trans_nona[["transport_durations"]].apply(lambda x: len(x))

In [ ]:
sns.histplot(data=df_trans_nona, x="number_of_transit_transfers")
plt.grid()

In [ ]:
sns.scatterplot(data=df_trans_nona, x="number_of_transit_transfers", y="duration_seconds")

In [ ]:
df_trans_nona["waiting_time"] = df_trans_nona["duration_seconds"] - df_trans_nona["transport_durations"].apply(lambda x: sum(x))
df_trans_nona["waiting_time"].hist(bins=50)

In [ ]:
df_trans_nona["change_time"] = 0
for i, row in df_trans_nona.iterrows():
    t = 0
    for a1,a2 in zip( row["transport_durations"][1:-1], row["transport_modes"][1:-1]):
        if a2 == "WALK":
            t = t + a1
    if t > 0:
        df_trans_nona.at[i, "change_time"] = t
df_trans_nona["change_time"].hist(bins=50)

In [ ]:
df_trans_nona["onboard_time"] = 0.0
for i, row in df_trans_nona.iterrows():
    t = 0
    for a1,a2 in zip( row["transport_durations"], row["transport_modes"]):
        if a2 == "BUS":
            t = t + a1
    if t > 0:
        df_trans_nona.at[i, "onboard_time"] = t
df_trans_nona["onboard_time"].hist(bins=50)

In [ ]:
sns.pairplot(
    data=df_trans_nona[["change_time", "waiting_time", "onboard_time", "walk_time_seconds", "duration_seconds", "number_of_transit_transfers"]],
    hue="number_of_transit_transfers",
    diag_kind="hist"
)

In [ ]:
df_trans_nona[["change_time", "waiting_time", "onboard_time", "walk_time_seconds", "duration_seconds"]].isna().apply(sum, axis=0)

# CAR AND TRANSIT

In [ ]:
# Check diff between car and transit
df_compare = pd.merge(
    df_car_park,
    df_trans,
    suffixes=["_carPark", "_trans"],
    how="outer",
    on=['row_index', 'origin_lat', 'origin_lon', 'from', 'dest_lat', 'dest_lon','to']
)

print("N. NAs in trans:", int(df_compare["duration_seconds_trans"].isna().sum()))
print("N. NAs in carPark:", int(df_compare["duration_seconds_carPark"].isna().sum()))

print("N. success in trans:", sum(df_compare["status_trans"]=="success"))
print("N. success in carPark:", sum(df_compare["status_carPark"]=="success"))

df_compare = df_compare[df_compare["status_trans"]=="success"]
df_compare.head()

In [ ]:
sns.pairplot(
    data=df_compare[["duration_seconds_carPark", "duration_seconds_trans"]],
    kind="reg"
)

# Not allBologna, but restrictedAv

In [ ]:
zoi: str = "restrictedAv"  ## Choose between "allBologna" and "restrictedAv"
method: str = "simplified" ## Choose between "simplified" and "extended"
version: str = "20250930"

In [ ]:
df_car_trans_di = project.get_dataitem(f"otp_results_{method}_{zoi}_CAR_PARK_TRANSIT_v{version}")
df_car_trans = df_car_trans_di.as_df()

df_car_di = project.get_dataitem(f"otp_results_{method}_{zoi}_CAR_v{version}")
df_car = df_car_di.as_df()

df_car_park_di = project.get_dataitem(f"otp_results_{method}_{zoi}_CAR_PARK_v{version}")
df_car_park = df_car_park_di.as_df()

df_trans_di = project.get_dataitem(f"otp_results_{method}_{zoi}_TRANSIT_v{version}")
df_trans = df_trans_di.as_df()

print(df_car.columns)

In [ ]:
df_car_park["status"].value_counts()

In [ ]:
print(df_car["status"].value_counts())
df_car[df_car["status"]=="success"]

In [ ]:
print(df_car_trans["status"].value_counts())
df_car_trans[df_car_trans["status"]=="success"].head()

In [ ]:
#Confronta car park con car park transit nel caso di AV ristretta -> comparabili? Competitivi?
df_compare = pd.merge(
    df_car_trans,
    df_car_park,
    suffixes=["_carParkTrans", "_carPark"],
    how="outer",
    on=['row_index', 'origin_lat', 'origin_lon', 'from', 'dest_lat', 'dest_lon','to']
)

print("N. NAs in carParkTrans:", int(df_compare["duration_seconds_carParkTrans"].isna().sum()))
print("N. NAs in carPark:", int(df_compare["duration_seconds_carPark"].isna().sum()))

print("N. success in carParkTrans:", sum(df_compare["status_carParkTrans"]=="success"))
print("N. success in carPark:", sum(df_compare["status_carPark"]=="success"))

df_compare = df_compare[df_compare["status_carParkTrans"]=="success"]
df_compare.head()


In [ ]:
sns.histplot(data=df_compare, x="duration_seconds_carParkTrans")
sns.histplot(data=df_compare, x="duration_seconds_carPark")
plt.legend(labels=["carParkTrans", "carPark"], title="Mode")

In [ ]:
sns.regplot(data=df_compare, x="duration_seconds_carParkTrans", y="duration_seconds_carPark")
plt.grid()